# Normalize M vector field in a VTK file

Reads a VTK file containing a vector field (`Mx`, `My`, `Mz`) defined on points, normalizes each vector by its local magnitude `|M|`, and writes out a new VTK file with:

- **`M`** — the original (unnormalized) vector field, packed as a 3-component vector array
- **`M_norm`** — the normalized vector field (unit vectors, `|M_norm| = 1` everywhere `M != 0`)
- **`M_magnitude`** — scalar field of `|M|` at each point (handy for coloring in ParaView)

Requires `pyvista` (`pip install pyvista`).

In [1]:
# If needed, install pyvista (uncomment the line below)
# %pip install pyvista

import numpy as np
import pyvista as pv

## 1. Set your file paths and array names

- `INPUT_PATH` / `OUTPUT_PATH`: your input and desired output VTK files
- If `Mx`, `My`, `Mz` are stored as **separate scalar arrays**, set `MX_NAME`, `MY_NAME`, `MZ_NAME` to match your file
- If they're already a **single 3-component vector array** (e.g. named `"M"`), set `VECTOR_NAME` to that name instead and it will be used in preference to the component names

In [5]:
sample = '452'
INPUT_PATH = f"data_dir/{sample}/{sample}_cropped_original_ascii_5thick.vtk"
#INPUT_PATH = f"data_dir/{sample}/665_cropped_og_22AUG26.vtk"

OUTPUT_PATH = f"data_dir/{sample}/{sample}_cropped_normalized_ascii_5thick.vtk"

# Used only if VECTOR_NAME is None
MX_NAME = "X"
MY_NAME = "Y"
MZ_NAME = "Z"

# If Mx/My/Mz are already a single 3-component vector array, put its name here
# (e.g. VECTOR_NAME = "M"). Leave as None to use MX_NAME/MY_NAME/MZ_NAME instead.

if sample == '105': 
    VECTOR_NAME = "magnetization"

elif sample == '452': 
    VECTOR_NAME = "magnetization"

elif sample == '665': 
    VECTOR_NAME = "magnetization"
else:
    VECTOR_NAME = "M"

EPS = 1e-30  # threshold below which |M| is treated as zero (avoids divide-by-zero)

## 2. Load the mesh and inspect available arrays

In [6]:
mesh = pv.read(INPUT_PATH)
print(f"Loaded {mesh.n_points} points from '{INPUT_PATH}'")
print("Available point-data arrays:", list(mesh.point_data.keys()))
mesh

Loaded 2805005 points from 'data_dir/452/452_cropped_original_ascii_5thick.vtk'
Available point-data arrays: ['magnetization']


ImageData (0x1ace36ab880)
  N Cells:      2238016
  N Points:     2805005
  X Bounds:     -3.740e+02, 3.740e+02
  Y Bounds:     -3.740e+02, 3.740e+02
  Z Bounds:     -2.000e+00, 2.000e+00
  Dimensions:   749, 749, 5
  Spacing:      1.000e+00, 1.000e+00, 1.000e+00
  N Arrays:     1

## 3. Build the (N, 3) array of M vectors

In [7]:
point_data = mesh.point_data

if VECTOR_NAME is not None:
    if VECTOR_NAME not in point_data:
        raise KeyError(
            f"Vector array '{VECTOR_NAME}' not found. "
            f"Available point arrays: {list(point_data.keys())}"
        )
    M = np.asarray(point_data[VECTOR_NAME])
    if M.ndim != 2 or M.shape[1] != 3:
        raise ValueError(
            f"Array '{VECTOR_NAME}' is not an (N, 3) vector array (got shape {M.shape})."
        )
else:
    missing = [n for n in (MX_NAME, MY_NAME, MZ_NAME) if n not in point_data]
    if missing:
        raise KeyError(
            f"Could not find component array(s) {missing}. "
            f"Available point arrays: {list(point_data.keys())}"
        )
    Mx = np.asarray(point_data[MX_NAME])
    My = np.asarray(point_data[MY_NAME])
    Mz = np.asarray(point_data[MZ_NAME])
    M = np.column_stack([Mx, My, Mz])

M.shape

(2805005, 3)

## 4. Compute magnitude and normalize

In [8]:
magnitude = np.linalg.norm(M, axis=1)
print(magnitude.shape)
print(M.shape)

safe_magnitude = np.where(magnitude < EPS, 1.0, magnitude)  # avoid 0/0
M_norm = M / safe_magnitude[:, None]
M_norm[magnitude < EPS] = 0.0  # zero out points where |M| was ~0

print(f"Magnitude range: [{magnitude.min():.6g}, {magnitude.max():.6g}]")
print(f"Number of near-zero points (|M| < {EPS}): {(magnitude < EPS).sum()}")

(2805005,)
(2805005, 3)
Magnitude range: [0.00176246, 1.81511]
Number of near-zero points (|M| < 1e-30): 0


## 5. Attach results to the mesh and save

In [9]:
mesh.point_data["M"] = M
mesh.point_data["M_norm"] = M_norm
mesh.point_data["M_magnitude"] = magnitude

mesh.save(OUTPUT_PATH)
print(f"Wrote normalized volume to '{OUTPUT_PATH}'")
print("Arrays written: 'M' (original vector), 'M_norm' (unit vector), 'M_magnitude' (scalar).")

Wrote normalized volume to 'data_dir/452/452_cropped_normalized_ascii_5thick.vtk'
Arrays written: 'M' (original vector), 'M_norm' (unit vector), 'M_magnitude' (scalar).


## 6. (Optional) Quick sanity-check plot in the notebook

Plots the mesh colored by `M_magnitude` with `M_norm` glyphs, using PyVista's Jupyter backend.

In [23]:
# #Uncomment to visualize directly in the notebook (requires trame: pip install trame trame-vtk trame-vuetify)
# pv.set_jupyter_backend("trame")
# plotter = pv.Plotter()
# plotter.add_mesh(mesh, scalars="M_magnitude", cmap="viridis")
# glyphs = mesh.glyph(orient="M_norm", scale=False, factor=0.5)
# plotter.add_mesh(glyphs, color="white")
# plotter.show()